In [4]:
import numpy as np
import pandas as pd
import os
import wandb
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize
import torch
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
import gc

In [ ]:
os.environ["WANDB_API_KEY"] = "API Key"
wandb.login()

wandb: Currently logged in as: nk23041720 (nk23041720-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
wandb_config = {
    "model_name": "google/muril-large-cased",
    "task": "nepali_multiclass_classification",
    "learning_rate": 2e-5,  
    "batch_size": 8,
    "gradient_accumulation_steps": 2,  
    "num_epochs": 10,  
    "max_length": 256,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-8, 
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "max_grad_norm": 1.0,  
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1,
    "random_seed": 42,
    "warmup_ratio": 0.1,  
    "eval_steps": 1000,  
    "logging_steps": 100,
    "save_steps": 1000,
    "early_stopping_patience": 30,
    "label_smoothing": 0.1, 
    "fp16": False,  
     "bf16": False,
}

In [4]:
run = wandb.init(
    project="topic-modeling-25k-experiments",
    name="muril(l)-25k-2e5-256-16-10",
    config=wandb_config,
    tags=["nepali", "bert", "classification", "muril"]
)

In [8]:
df = pd.read_excel("/home/rupak/Desktop/Topic-Modeling /dataset/topic-modeling-25k-dataset.xlsx")
df.head()

,sentenceid,relevant_sentences,domainid
0,COM_D3E_S1_S2_6369,अक्षय कोषको आकार एक करोड रूपिया पुर्‍याउने तत्...,D3E
1,COM_D3E_S1_S2_4941,जेहेन्दार विद्यार्थीलाई छात्रवृत्ति प्रदान गरि...,D3E
2,COM_D3E_S1_S2_1399,अक्षराङ्कन पद्धतिमा शुरूमा आठ ग्रेडको व्यवस्था...,D3E
3,COM_D3E_S1_S2_1390,अक्षराङ्कन लागू हुनु पहिला विद्यार्थी दुई कित्...,D3E
4,COM_D3E_S1_S2_1049,अक्सिटोसिनले अध्यात्मिक सोच बढाउनमा सहायक पाइए...,D3E


In [9]:
df = df.sample(frac=1, random_state=wandb.config.random_seed).reset_index(drop=True)
df.head()

Error: You must call wandb.init() before wandb.config.random_seed

In [10]:
unique_labels = sorted(df["domainid"].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
df["labels"] = df["domainid"].map(label2id)

In [8]:
dataset = Dataset.from_pandas(df[['relevant_sentences', 'labels']])
dataset = dataset.rename_column('relevant_sentences', 'text')

In [9]:
dataset.shape

(25006, 2)

In [10]:
train_test_split = dataset.train_test_split(
    test_size=0.2,
    seed=wandb.config.random_seed,
    shuffle=True
)
train_dataset = train_test_split['train']

In [11]:
val_test_split = train_test_split['test'].train_test_split(
    test_size=0.5,
    seed=wandb.config.random_seed,
    shuffle=True
)
val_dataset = val_test_split['train']  # 10% of total
test_dataset = val_test_split['test']   # 10% of total

In [12]:
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [13]:
model_name = wandb.config.model_name
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [14]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=False,  # Dynamic padding with data collator
        truncation=False,
        max_length=wandb.config.max_length
    )

In [15]:
dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])

Map:   0%|          | 0/20004 [00:00<?, ? examples/s]

Map:   0%|          | 0/2501 [00:00<?, ? examples/s]

Map:   0%|          | 0/2501 [00:00<?, ? examples/s]

In [16]:
num_labels = len(unique_labels)
num_labels

5

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.torch.float32,
    device_map="auto",
    trust_remote_code=False,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-large-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

In [19]:
total_steps = (len(dataset["train"]) // (wandb.config.batch_size * wandb.config.gradient_accumulation_steps)) * wandb.config.num_epochs
warmup_steps = int(total_steps * wandb.config.warmup_ratio)

In [20]:
training_args = TrainingArguments(
    output_dir="/home/rupak/Desktop/Topic-Modeling /topic modeling 25k/checkpoint/muril(l)-25k-2e5-256-16-10",
    eval_strategy="steps",
    eval_steps=wandb.config.eval_steps,
    save_strategy="steps",
    save_steps=wandb.config.save_steps,
    learning_rate=wandb.config.learning_rate,
    per_device_train_batch_size=wandb.config.batch_size,
    per_device_eval_batch_size=wandb.config.batch_size,
    gradient_accumulation_steps=wandb.config.gradient_accumulation_steps,
    num_train_epochs=wandb.config.num_epochs,
    weight_decay=wandb.config.weight_decay,
    adam_beta1=wandb.config.adam_beta1,
    adam_beta2=wandb.config.adam_beta2,
    adam_epsilon=wandb.config.adam_epsilon,
    max_grad_norm=wandb.config.max_grad_norm,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_weighted",
    greater_is_better=True,
    save_total_limit=3,
    label_smoothing_factor=wandb.config.label_smoothing,
    # Enhanced logging
    logging_dir='./logs',
    logging_steps=wandb.config.logging_steps,
    logging_first_step=True,
    # Wandb integration
    report_to="wandb",
    run_name=f"nepberta-optimized-{wandb.run.id}",
    # Full precision training (FP32)
    fp16=wandb.config.fp16,
    bf16=wandb.config.bf16,
    dataloader_drop_last=False,
    eval_accumulation_steps=None,
    # Additional optimization
    dataloader_num_workers=2,
    remove_unused_columns=False,
    push_to_hub=False,
)

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Get probabilities for AUROC
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    # Calculate comprehensive metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )

    # Macro averages for unbiased evaluation
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    # Calculate AUROC (for multiclass)
    if num_labels > 2:
        labels_binarized = label_binarize(labels, classes=range(num_labels))
        try:
            auroc_weighted = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='weighted')
            auroc_macro = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='macro')
        except ValueError:
            auroc_weighted = 0.0
            auroc_macro = 0.0
    else:
        auroc_weighted = roc_auc_score(labels, probabilities[:, 1])
        auroc_macro = auroc_weighted

    metrics = {
        "accuracy": accuracy,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "auroc_weighted": auroc_weighted,
        "auroc_macro": auroc_macro
    }

    return metrics

In [22]:
class CustomTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_accuracy_history = []
        self._last_train_metrics = {}



    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss and optionally calculate training accuracy
        """
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss = outputs.get("loss")

        # Calculate training accuracy periodically
        if (self.state.global_step % (self.args.logging_steps * 2) == 0 and
            labels is not None and self.state.global_step > 0):
            with torch.no_grad():
                predictions = torch.argmax(outputs.logits, dim=-1)
                accuracy = (predictions == labels).float().mean().item()
                self._last_train_metrics = {"train_accuracy": accuracy}

        return (loss, outputs) if return_outputs else loss

In [23]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=wandb.config.early_stopping_patience)]
)

/tmp/ipykernel_729079/3450341322.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [24]:
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision Macro,Recall Macro,F1 Macro,Auroc Weighted,Auroc Macro
1000,0.320100,0.372407,0.886845,0.887796,0.886845,0.886377,0.888369,0.888177,0.887358,0.981878,0.982155
2000,0.264400,0.365662,0.899640,0.900051,0.899640,0.899272,0.900244,0.900867,0.899995,0.985669,0.985844
3000,0.173100,0.453348,0.899640,0.899895,0.899640,0.899400,0.900771,0.900741,0.900390,0.982394,0.982654
4000,0.090900,0.536194,0.898041,0.898353,0.898041,0.897656,0.898621,0.899500,0.898531,0.984352,0.984508
5000,0.137900,0.585456,0.896042,0.897499,0.896042,0.895329,0.897328,0.898022,0.896266,0.978203,0.978550
6000,0.075700,0.612755,0.901240,0.901612,0.901240,0.901237,0.902141,0.902181,0.901971,0.980655,0.980840
7000,0.026300,0.772250,0.900440,0.901259,0.900440,0.899728,0.900900,0.902223,0.900470,0.975357,0.975737
8000,0.024900,0.770871,0.895242,0.895700,0.895242,0.894999,0.896025,0.896604,0.895842,0.975566,0.975765
9000,0.015100,0.819970,0.896441,0.896503,0.896441,0.896148,0.896978,0.897630,0.896992,0.973710,0.974013
10000,0.006400,0.859529,0.899240,0.899061,0.899240,0.898958,0.899434,0.900484,0.899768,0.974039,0.974232


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


TrainOutput(global_step=12510, training_loss=0.13035789202611378, metrics={'train_runtime': 3946.5321, 'train_samples_per_second': 50.688, 'train_steps_per_second': 3.17, 'total_flos': 1.1924197222044912e+16, 'train_loss': 0.13035789202611378, 'epoch': 10.0})

In [25]:
test_results = trainer.evaluate(eval_dataset=dataset["test"])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
for key, value in test_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').replace('_', ' ').title()
        print(f"{metric_name:<20}: {value:.4f}")

Loss                : 0.5354
Accuracy            : 0.9056
Precision Weighted  : 0.9068
Recall Weighted     : 0.9056
F1 Weighted         : 0.9060
Precision Macro     : 0.9052
Recall Macro        : 0.9044
F1 Macro            : 0.9046
Auroc Weighted      : 0.9836
Auroc Macro         : 0.9834
Runtime             : 6.1637
Samples Per Second  : 405.7650
Steps Per Second    : 50.7810


In [27]:
# Get detailed predictions for test set
test_predictions = trainer.predict(dataset["test"])
test_logits = test_predictions.predictions
test_labels = test_predictions.label_ids
test_pred_labels = np.argmax(test_logits, axis=-1)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
# Convert back to original domain labels for interpretability
test_pred_domains = [id2label[pred] for pred in test_pred_labels]
test_true_domains = [id2label[label] for label in test_labels]

In [29]:
report = classification_report(
    test_labels, 
    test_pred_labels, 
    target_names=[id2label[i] for i in range(num_labels)],
    digits=4
)

In [30]:
print(report)

              precision    recall  f1-score   support

         D1A     0.9407    0.9231    0.9318       533
         D2H     0.8905    0.8868    0.8887       486
         D3E     0.8969    0.9241    0.9103       527
         D4C     0.9685    0.9294    0.9486       496
         D5G     0.8295    0.8584    0.8437       459

    accuracy                         0.9056      2501
   macro avg     0.9052    0.9044    0.9046      2501
weighted avg     0.9068    0.9056    0.9060      2501



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd

# 1. Load checkpoint (labels are automatically loaded from config)
checkpoint_path =  "topic-modeling-25k/checkpoint/nepberta-25k-2e5-256-16-10/checkpoint-9000"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)


# Extract label mappings from model config
id2label = model.config.id2label
label2id = model.config.label2id
num_labels = model.config.num_labels

print(f"Labels: {id2label}")
print(f"Number of labels: {num_labels}\n")

# 2. Load and prepare test dataset
df = pd.read_excel("/home/rupak/Desktop/Topic-Modeling /dataset/topic-modeling-25k-dataset.xlsx")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df["labels"] = df["domainid"].map(label2id)

# Create test split (last 10%)
test_size = int(len(df) * 0.1)
test_df = df.tail(test_size)
test_dataset = Dataset.from_pandas(test_df[['relevant_sentences', 'labels']])
test_dataset = test_dataset.rename_column('relevant_sentences', 'text')

# Tokenize
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=['text'])

# 3. Get predictions
trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="./temp", per_device_eval_batch_size=8)
)

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# 4. Create confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# 5. Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_names = [id2label[i] for i in range(num_labels)]

# Counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12, fontweight='bold')

# Percentages
sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], square=True, linewidths=0.5, vmin=0, vmax=100)
axes[1].set_title('Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('True Label', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 6. Print metrics
print("\nCONFUSION MATRIX (Counts):")
print(pd.DataFrame(cm, index=class_names, columns=class_names))

print("\nNORMALIZED CONFUSION MATRIX (%):")
print(pd.DataFrame(cm_normalized.round(2), index=class_names, columns=class_names))

# Per-class metrics
print("\nPER-CLASS METRICS:")
for i, name in enumerate(class_names):
    tp = cm[i, i]
    precision = tp / cm[:, i].sum() if cm[:, i].sum() > 0 else 0
    recall = tp / cm[i, :].sum() if cm[i, :].sum() > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"{name}: P={precision:.4f}, R={recall:.4f}, F1={f1:.4f}, Support={cm[i, :].sum()}")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/home/rupak/topic_modeling_25k/checkpoint/nepberta-25k-2e5-256-16-10/checkpoint-9000'. Use `repo_type` argument if needed.